# DVC Quickstart Assignment

Welcome! In this notebook, you'll get hands-on experience with DVC, a tool for versioning datasets and models.

**Objective:** Track a dataset and a model file using DVC in under 10 minutes.

## 1. Initialize DVC
Let's initialize DVC in your project folder. This will create the necessary configuration files. You must comment DVC part in .gitignore.

In [ ]:
#!pip install -q dvc
#!git config --global user.email "you@example.com"
#!git config --global user.name "Your Name"

In [ ]:
# Initialize DVC
!git init
!dvc init

## 2. Track a Dataset with DVC
Let's add a sample dataset to DVC tracking.

In [ ]:
import pandas as pd
import numpy as np

n_rows = 2_000

print(f"Création d'un dataset de {n_rows} lignes.")

# Générer les données synthétiques
# np.random.seed(42)  # Pour la reproductibilité
df = pd.DataFrame({
    'feature1': np.random.randint(0, 1000, size=n_rows),
    'feature2': np.random.randn(n_rows),           # float64
    'feature3': np.random.choice(['A', 'B', 'C', 'D'], size=n_rows),
    'feature4': np.random.uniform(0, 1, size=n_rows),
    'feature5': np.random.exponential(1.0, size=n_rows)
})

# Sauvegarder en CSV
df.to_csv('sample_dataset.csv', index=False)

# Vérifier la taille
import os
size_mb = os.path.getsize('sample_dataset.csv') / (1024 * 1024)
print(f"Fichier créé : sample_dataset.csv ({size_mb:1f} Mo)")

We don't want to push data that is too large (here ~150MB) => dvc solves this problem.

exemple error after pushing :
"remote: error: File large_dataset.csv is 159.77 MB; this exceeds GitHub's file size limit of 100.00 MB"

In [ ]:
# Track the dataset file with DVC
!dvc add sample_dataset.csv

In [ ]:
# DVC tracks changes to this dataset
df.rename(columns={'feature3': 'feature7'}, inplace=True)
df.to_csv('sample_dataset.csv', index=False)

In [ ]:
# Let's check if dvc can detect the change
!dvc diff

Why there are no diffs to show ? \
Because DVC works like Git, it needs a reference version to compare the file against... To do that, we need to track the .dvc file with Git

In [ ]:
# Our csv file have been changed so we should update it on dvc
!dvc add sample_dataset.csv
!git add sample_dataset.csv.dvc
!git commit -m "add dataset"

## 3. Track a Model File with DVC
Let's create a simple model file and add it to DVC tracking.

In [ ]:
# Create a simple model file
with open('model.pkl', 'wb') as f:
    f.write(b'Model binary content')
print('model.pkl created')

In [ ]:
# Track the model file with DVC
!dvc add model.pkl

## 4. Check Git & DVC Status
Let's check the status of tracked files.

In [ ]:
# Check DVC status for tracked files
!dvc status
!git status

model.pkl.dvc is still not tracked by git, let's do it and go further.


In [ ]:
!git add model.pkl.dvc
!dvc config core.autostage true

End of this quickstart: dvc files are now automatically stagged in git.

## 5. Modify the Dataset and Track Changes
Let's make another modification to the dataset and see how DVC detects it.

In [ ]:
# Read the current dataset
df = pd.read_csv('sample_dataset.csv')
print(f"Current dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

In [ ]:
# Modify the dataframe - add a new column and filter some rows
df['feature6'] = df['feature1'] * df['feature4']  # New computed column
df = df[df['feature1'] > 200]  # Filter rows where feature1 > 200

print(f"Modified dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# Save the modified dataset
df.to_csv('sample_dataset.csv', index=False)
print("Dataset modified and saved!")

In [ ]:
# Check what DVC detected as changed
!dvc diff --json --show-hash

In [ ]:
# Track the new version with DVC
!dvc add sample_dataset.csv

## 6. Commit Changes to Git
DVC metadata files are already staged in Git. The actual data stays in DVC cache.

In [ ]:
# Check git status to see what files changed
!git status

In [ ]:
# Stage the DVC files (not the actual data!)
!git add .gitignore
!git status

In [ ]:
# Commit the changes
!git commit -m "Update dataset: added feature6 column and filtered rows"

# **Optional: Code and data remote repositories**

# Github and Dagshub

As well as git with github, dvc can be associate to a remote service storage nammed DagsHub. \
If you have a remote repository configured, you can create a dagshub repo linked to the github one and then push your changes on both.

1.   Go to [dagshub](https://dagshub.com/user/sign_up?) and create your account with your github account
2.   Click on create/new repository, then choose connect a repository.
3.   Congratulations, Your github and dagshub repos are now linked together !



# Link local DVC and Dagshub repo

Go to your dagshub repo and click on the green button nammed "data". \
Here you can get all the commands to link your repo with differents services as DagsHub Client, DVC and S3 Bucket. We're only interested by the dvc part. \
Execute in your shell the commands from sections: "Add DVC remote", "Setup credentials" \
You can execute the following command to make dagshub as your default remote storage for the project
```bash
# Set my remote as default
dvc remote default origin
# Stagged the dvc config file
git add .dvc
# Make sure that .dvc/config.local is not tracked by git
git status
```

Now we can push our data to the repo, ".dvc" files are tracked by git so they're uploaded to github, datas are uploaded to dagshub.

```bash
# Commit change
git commit -m "update DVC config"
# Push to remote repository
git push origin main
# Push to remote repository
dvc push
```

**Note:** The actual data file (`sample_dataset.csv`) is NOT pushed to Git because it's in `.gitignore`. Only the metadata file (`sample_dataset.csv.dvc`) is pushed, which contains the hash of the data version.